# MARITIME-X · 01 Cluster and pipeline verification

Checks the 3-node Hadoop / YARN / Spark cluster through its REST APIs and shows the metrics each Spark stage recorded on the real AIS dataset. Start the cluster first: `.\scripts\cluster.ps1 start`.

In [1]:
import json, urllib.request
import pandas as pd
from pymongo import MongoClient

def get(url):
    with urllib.request.urlopen(url, timeout=15) as r:
        return json.load(r)

db = MongoClient('mongodb://localhost:27017')['maritime_x']

## HDFS: NameNode, DataNodes, blocks and replication

In [2]:
q = 'http://localhost:9870/jmx?qry=Hadoop:service=NameNode,name='
st = get(q + 'FSNamesystemState')['beans'][0]
fs = get(q + 'FSNamesystem')['beans'][0]
info = get(q + 'NameNodeInfo')['beans'][0]
print('state', st['FSState'], '| live DataNodes', st['NumLiveDataNodes'], '| dead', st['NumDeadDataNodes'])
print('files', st['FilesTotal'], '| blocks', st['BlocksTotal'], '| under-replicated', fs['UnderReplicatedBlocks'], '| missing', fs['MissingBlocks'])
pd.DataFrame([{'datanode': v['xferaddr'], 'used_GB': round(v['used'] / 1e9, 2), 'blocks': v['numBlocks'], 'state': v['adminState']}
              for v in json.loads(info['LiveNodes']).values()])

state Operational | live DataNodes 2 | dead 0
files 2493 | blocks 2410 | under-replicated 0 | missing 0


,datanode,used_GB,blocks,state
0,127.0.0.1:9966,17.71,2410,In Service
1,127.0.0.1:9866,17.71,2410,In Service


In [3]:
raw = get('http://localhost:9870/webhdfs/v1/maritime/raw/ais/marine_ais_dataset.csv?op=GETFILESTATUS')['FileStatus']
print(f"raw AIS file: {raw['length']:,} bytes, replication {raw['replication']}, block size {raw['blockSize'] // 2**20} MB, "
      f"{-(-raw['length'] // raw['blockSize'])} blocks")
lake = get('http://localhost:9870/webhdfs/v1/maritime?op=GETCONTENTSUMMARY')['ContentSummary']
print(f"/maritime lake: {lake['fileCount']:,} files, {lake['length'] / 1e9:.1f} GB logical, {lake['spaceConsumed'] / 1e9:.1f} GB on disk (x replication)")

raw AIS file: 4,899,356,398 bytes, replication 2, block size 128 MB, 37 blocks


/maritime lake: 2,134 files, 17.2 GB logical, 34.3 GB on disk (x replication)


## YARN: ResourceManager, NodeManagers and Spark applications

In [4]:
m = get('http://localhost:8088/ws/v1/cluster/metrics')['clusterMetrics']
print('active NodeManagers', m['activeNodes'], '| memory', m['totalMB'], 'MB | vcores', m['totalVirtualCores'], '| apps completed', m['appsCompleted'])
pd.DataFrame([{'node': n['id'], 'state': n['state'], 'memory_MB': n['usedMemoryMB'] + n['availMemoryMB'], 'containers': n['numContainers']}
              for n in get('http://localhost:8088/ws/v1/cluster/nodes')['nodes']['node']])

active NodeManagers 2 | memory 6144 MB | vcores 12 | apps completed 0


,node,state,memory_MB,containers
0,kubernetes.docker.internal:45454,RUNNING,3072,0
1,kubernetes.docker.internal:45464,RUNNING,3072,0


YARN keeps finished applications in memory, so this list is empty after a ResourceManager restart; the `spark_master` column in the pipeline metrics below records where each stage ran.

In [5]:
apps = (get('http://localhost:8088/ws/v1/cluster/apps')['apps'] or {}).get('app', [])
if not apps: print('no applications since the last ResourceManager start')
pd.DataFrame([{'id': a['id'], 'name': a['name'], 'state': a['state'], 'final': a['finalStatus'],
               'minutes': round(a['elapsedTime'] / 60000, 1), 'MB-seconds': a['memorySeconds']} for a in apps])

no applications since the last ResourceManager start


""


## Spark standalone master and workers

In [6]:
sm = get('http://localhost:8080/json/')
print(sm['url'], sm['status'], 'alive workers:', sm['aliveworkers'], 'cores:', sm['cores'])
pd.DataFrame([{'worker': w['id'], 'host': f"{w['host']}:{w['port']}", 'state': w['state'], 'cores': w['cores'], 'memory_MB': w['memory']} for w in sm['workers']])

spark://localhost:7077 ALIVE alive workers: 2 cores: 8


,worker,host,state,cores,memory_MB
0,worker-20261007103334-localhost-7079,localhost:7079,ALIVE,4,2048
1,worker-20261007103333-localhost-7078,localhost:7078,ALIVE,4,2048


## Pipeline metrics recorded by each Spark stage

These values are computed by Spark during the run and stored in HDFS (`/maritime/output/pipeline_metrics`) and MongoDB. `spark_master` shows where each stage ran.

In [7]:
rows = []
for d in db.pipeline_metrics.find({}, {'_id': 0}):
    rows.append({'stage': d['stage'], 'spark_master': d.get('spark_master'), 'elapsed_s': d.get('elapsed_seconds'), 'recorded_at': d.get('recorded_at')})
pd.DataFrame(rows).sort_values('recorded_at')

,stage,spark_master,elapsed_s,recorded_at
0,clean_ais,yarn,1104.2,2026-10-06T18:03:50Z
1,trajectories,yarn,588.8,2026-10-06T18:13:49Z
2,port_visits,yarn,348.2,2026-10-06T18:19:48Z
5,port_congestion,yarn,73.5,2026-10-06T18:21:14Z
9,turnaround,yarn,9.7,2026-10-06T18:21:33Z
4,ais_gaps,yarn,456.2,2026-10-06T18:29:19Z
6,gap_kmeans,yarn,108.8,2026-10-06T18:34:44Z
7,risk_scoring,yarn,26.9,2026-10-06T18:35:20Z
8,identity,yarn,311.6,2026-10-07T04:33:31Z
10,serving_views,yarn,605.4,2026-10-07T04:43:41Z


In [8]:
c = db.pipeline_metrics.find_one({'stage': 'clean_ais'}, {'_id': 0})
{k: c[k] for k in ('raw_rows', 'clean_rows', 'removed_rows', 'invalid_mmsi_rows', 'vessels', 'window_start_epoch', 'window_end_epoch')}

{'raw_rows': 47958624,
 'clean_rows': 47912404,
 'removed_rows': 46220,
 'invalid_mmsi_rows': 36765,
 'vessels': 24529,
 'window_start_epoch': 1735689600,
 'window_end_epoch': 1736294399}